In [ ]:
import sys, os
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))  # scrape_code/
sys.path.insert(0, os.path.join(_ROOT, 'scrapers'))
sys.path.insert(0, os.path.join(_ROOT, 'utils'))

## General Maintenance Notebook for the Pipeline and the database 

In [1]:
import numpy as np 
import pandas as pd 
from scraper import scrape_model 
import os, sys, time
import requests 
import datetime, dateparser

from bs4 import BeautifulSoup
from tqdm import tqdm


from selenium import webdriver 
from selenium.webdriver.chrome.options import Options 
from selenium.webdriver.common.by import By 
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from util_data_funcs import *


def close_out_driver(wd):
    wd.close()
    wd.quit()


headers = {
    'User-Agent': 
           'Mozilla/5.0 (X11; Linux x86_64)'+\
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}


def parse_game_date(date_parse):    
    hour = date_parse[:date_parse.find(':')]
    minute = date_parse[date_parse.find(':')+1:date_parse.find(' ')]
    am_pm = date_parse[date_parse.find(' ')+1:date_parse.find(',')]

    month_day = date_parse[date_parse.find(',')+2:-6]
    month, day = datetime.datetime(month_day.split(' ')[0], '%b').month , month_day.split(' ')[1]
    # year = date_parse[-4:]

    return datetime.date(int(month), int(day))


def parse_game_date_v2(date_parse):
    dow = date_parse[:date_parse.find(',')]
    month = date_parse[date_parse.find(',')+2:date_parse.find(',')+5]
    month_num = datetime.datetime.strptime(month, '%b').month
    dom = date_parse[date_parse.find(month)+4:]
    return (int(month_num), int(dom))


def parse_game_date_v3(df):
    fr_parse = dateparser.parse(df['date'], languages=['fr'])
    es_parse = dateparser.parse(df['date'], languages=['es'])
    en_parse = dateparser.parse(df['date'], languages=['en'])

    if fr_parse == None:
        if es_parse == None:
            # print(df['date'])
            return en_parse.date().replace(year=df['season'])
        else:
            return es_parse.date().replace(year=df['season'])
    else:
        return fr_parse.date().replace(year=df['season'])


sched_path = 'formed_data/game_schedule_data'
sched_file_dir = os.listdir(sched_path)
urc_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'URC_' in i]
prem_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'Prem_' in i]
sr_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'SR_' in i]
t14_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'T14_' in i]
champ_cup_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'ChampCup' in i]

six_nations_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'SixNat_' in i]
rchamp_sched_files = [sched_path + '/' + i for i in sched_file_dir if 'RChamp_' in i]



game_path = 'formed_data/game_data'
game_file_dir = os.listdir(game_path)
urc_game_files = [game_path + '/' + i for i in game_file_dir if 'URC_' in i]
prem_game_files = [game_path + '/' + i for i in game_file_dir if 'Prem_' in i]
sr_game_files = [game_path + '/' + i for i in game_file_dir if 'SR_' in i]
t14_game_files = [game_path + '/' + i for i in game_file_dir if 'T14_' in i]
champ_cup_game_files = [game_path + '/' + i for i in game_file_dir if 'ChampCup_' in i]

six_nations_game_files = [game_path + '/' + i for i in game_file_dir if 'SixNat_' in i]
rchamp_game_files = [game_path + '/' + i for i in game_file_dir if 'RChamp_' in i]



t14_game_df, t14_player_df, t14_team_df = gather_dataframes('T14')
urc_game_df, urc_player_df, urc_team_df = gather_dataframes('URC')
prem_game_df, prem_player_df, prem_team_df = gather_dataframes('Prem')
champ_cup_game_df, champ_cup_player_df = gather_dataframes('ChampCup')

# sr_game_df, sr_player_df, sr_team_df = gather_dataframes('SR')
six_nat_game_df, six_nat_player_df, six_nat_team_df = gather_dataframes('SixNat')

# rchamp_game_df, rchamp_player_df, rchamp_team_df = gather_dataframes('RChamp')

/home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code/util_data_funcs.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grab_files = lambda league, order: pd.concat(list(map(pd.read_csv, league_dict[league][order])))
/home/colingrosh/Desktop/sports_analytics/code/rugby_v2/Rugby_DB_Analysis/scrape_code/util_data_funcs.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grab_files = lambda league, order: pd.concat(list(map(pd.read_csv, league_dict[league][order]

### Checking the number of Games in each league season and comparing to the games that I have pulled so far 

In [2]:
# Util function used in schedule pulls 
def label_table_parse(labels, table):
    row_step = []
    for row in table:
        val_text = row.text
        if val_text not in labels:
            row_step.append(val_text)

    row_ordered = [[row_step[i-1], row_step[i]] \
                    for i in range(1, len(row_step), 2)]

    return row_ordered 

# Takes the game and league to parse the match stats 
def get_match_stats(game_id, league_id): 

    # URL declaration, scraping, and initial parsing to get the tables on the page 
    url = 'https://www.espn.co.uk/rugby/matchstats/_/gameId/{}/league/{}'.format(
        game_id, league_id
    )
    response = requests.get(url, headers=headers).content
    soup = BeautifulSoup(response, 'html.parser')
    tables = soup.find_all('table')

    nav_bar_items = soup.find_all('nav')

    # Group labels 
    group_2_labels = ['Tries', 'Conversion Goals', 
                        'Penalty Goals', 'Kick Percent Success'
    ]
    group_3_labels = ['Kicks From Hand', 'Passes', 'Runs']
    group_4_labels = [
        'Possession 1H/2H', 'Territory 1H/2H', 'Clean Breaks', 
        'Defenders Beaten', 'Offload', 'Rucks Won', 
        'Mauls Won', 'Turnovers Conceded'
    ]
    group_8_labels = ['Red Cards', 'Yellow Cards', 'Total Free Kicks Conceded']

    # Parsed data lists used by multiple groups 
    four_tables = soup.find_all(
        class_='sub-module equal-height countChartList height-reset'
    )
    check_top_largeLabels = soup.find_all(
        class_="stat-graph compareLineGraph twoTeam largeLabels"
    )
    stacked_rls = soup.find_all(class_='stacked-rl')

    # GROUP 1: Home and Away team parsing 
    top_bar = soup.find(class_='competitors')

    h_a_team_bar = [
        top_bar.find(class_='team team-a'),
        top_bar.find(class_='team team-b')
    ]

    # GROUP 2: Match Events 
    match_event = four_tables[0].find('tbody')
    match_event_rows = match_event.find_all('td')

    group_2_ordered = label_table_parse(group_2_labels, match_event_rows)

    # GROUP 3: Kick/Pass/Run 
    home_away_total_meters = [
        int(i.text) for i in check_top_largeLabels[0].find_all(class_='chartValue')
    ]

    meter_rows = four_tables[1].find('tbody').find_all('td')
    group_3_ordered = label_table_parse(group_3_labels, meter_rows)

    # GROUP 4: Attacking 
    attack_rows = stacked_rls[0].find('tbody').find_all('td')
    group_4_ordered = label_table_parse(group_4_labels, attack_rows)

    # GROUP 5: Possession and Territory 
    terr_vals = soup.find_all(
        class_="stat-graph compareLineGraph twoTeam largeLabels large"
    )[0].find_all(class_='chartValue')

    poss_vals = check_top_largeLabels[1].find_all(class_='chartValue')

    # GROUP 6: Set Pieces 
    sp_charts = four_tables[2].find_all(class_='countChart')

    # list of scrums and then lineouts. 00 is home scrums, 10 is home lineouts 
    h_a_set_pieces = [
        [
            sp_charts[0].find_all(class_='countLabel')[0].text,
            sp_charts[0].find_all(class_='countLabel')[1].text
        ],
        [
            sp_charts[1].find_all(class_='countLabel')[0].text,
            sp_charts[1].find_all(class_='countLabel')[1].text
        ]
    ]

    # GROUP 7: Defending 
    # list of lists, raw tackles and then home tackles 
    tackles = [
        four_tables[3].find_all(class_='home-team'),
        four_tables[3].find_all(class_='away-team')
    ]

    # GROUP 8: Discipline and Penalties 
    disc_rows = tables[3].find('tbody').find_all('td')
    penalty = stacked_rls[1].find(class_='countChart').find_all(
        class_='countLabel'
    )

    group_8_ordered = label_table_parse(group_8_labels, disc_rows)

    # Combine all group variables 
    top_data_dict = {
        'game_id': game_id,
        'league_id': league_id,

        # GROUP 1 metrics 
        'home_team': h_a_team_bar[0].find(class_='short-name').text,
        'home_team_score': int(h_a_team_bar[0].find(class_='score-container').text), 
        'away_team': h_a_team_bar[1].find(class_='short-name').text,
        'away_team_score': int(h_a_team_bar[1].find(class_='score-container').text),

        # GROUP 2 metrics 
        'home_tries': group_2_ordered[0][0],
        'away_tries': group_2_ordered[0][1],
        'home_conversions': group_2_ordered[1][0],
        'away_conversions': group_2_ordered[1][1],
        'home_penalty_goals': group_2_ordered[2][0],
        'away_penalty_goals': group_2_ordered[2][1],
        'home_kick_percent': group_2_ordered[3][0],
        'away_kick_percent': group_2_ordered[3][1],

        # GROUP 3 metrics 
        'home_total_meters': home_away_total_meters[0],
        'away_total_meters': home_away_total_meters[1],
        'home_kfh': group_3_ordered[0][0],
        'away_kfh': group_3_ordered[0][1],
        'home_pass_meters': group_3_ordered[1][0],
        'away_pass_meters': group_3_ordered[1][1],
        'home_runs': group_3_ordered[2][0],
        'away_runs': group_3_ordered[2][1],

        # GROUP 4 metrics 
        'home_possession_1h_2h': group_4_ordered[0][0],
        'home_territory_1h_2h': group_4_ordered[1][0],
        'home_clean_breaks': group_4_ordered[2][0],
        'home_defenders_beaten': group_4_ordered[3][0],
        'home_offloads': group_4_ordered[4][0],
        'home_rucks_won': group_4_ordered[5][0],
        'home_mauls_won': group_4_ordered[6][0],
        'home_turnovers_conceeded': group_4_ordered[7][0],
        'away_possession_1h_2h': group_4_ordered[0][1],
        'away_territory_1h_2h': group_4_ordered[1][1],
        'away_clean_breaks': group_4_ordered[2][1],
        'away_defenders_beaten': group_4_ordered[3][1],
        'away_offloads': group_4_ordered[4][1],
        'away_rucks_won': group_4_ordered[5][1],
        'away_mauls_won': group_4_ordered[6][1],
        'away_turnovers_conceeded': group_4_ordered[7][1],

        # GROUP 5 metrics 
        'home_total_possession': poss_vals[0].text,
        'home_total_territory': terr_vals[0].text,
        'away_total_possesion': poss_vals[1].text,
        'away_total_territory': terr_vals[1].text,

        # GROUP 6 metrics 
        'home_scrum': h_a_set_pieces[0][0],
        'home_lineout': h_a_set_pieces[1][0],
        'away_scrum': h_a_set_pieces[0][1],
        'away_lineout': h_a_set_pieces[1][1],

        # GROUP 7 metrics 
        'home_tackles': tackles[0][0].text, 
        'home_tackle_perc': tackles[1][0].text, 
        'away_tackles': tackles[0][1].text, 
        'away_tackle_perc': tackles[1][1].text,

        # GROUP 8 metrics 
        'home_red_cards': group_8_ordered[0][0],
        'home_yellow_cards': group_8_ordered[1][0],
        'home_free_kicks_con': group_8_ordered[2][0], 
        'away_red_cards': group_8_ordered[0][1],
        'away_yellow_cards': group_8_ordered[1][1],
        'away_free_kicks_con': group_8_ordered[2][1],
        'home_penalties': int(penalty[0].text),
        'away_penalties': int(penalty[1].text)

    }

    df = pd.DataFrame(top_data_dict, index=[0])
    
    return df, nav_bar_items

# Functions below are used for gathering team and league data based on the inputs to the class 
def get_teams_list(season, league):
    
    test_url = 'https://www.espn.co.uk' + \
        '/rugby/table/_/league/{}/season/{}'.format(league, season)
    response = requests.get(test_url, headers=headers).content
    soup_team = BeautifulSoup(response, 'html.parser')

    get_index = lambda x, char: x.find(char)

    tbodies = soup_team.find_all('tbody')
    if len(tbodies) > 1:
        tbodies = tbodies[::2]

    team_name_link = {}
    for tbody in tbodies:

        row_trs = tbody.find_all('tr')

        for tr in row_trs:
            # print(tr.text)
            td_start = tr.find_all('td')[0]
            try:
                team_link = td_start.find_all('a')[0]['href']
                # print(team_link)
                # team_name_link[td_start.find_all('a')[1].find('span').text] = [
                # team_name_link[td_start.find_all('a')[2].text] = [
                team_name_link[td_start.find_all('a')[-1].text] = [
                    team_link,
                    team_link[team_link.find('id/')+3:team_link.find('id/')+3+\
                                get_index(team_link[team_link.find('id/')+3:], '/')]
                ]
            except:
                print(tr)
                team_link = None 
                team_name_link[td_start.find_all('a')[2].text] = [
                    team_link,
                    None
                ]

    return team_name_link 
        

def get_schedule(teams_list, league, season, team_input = None):
    
    if len(teams_list.keys()) == 0:
        sched_dict = {}
        
    elif team_input is not None:
        results_base_url = "https://www.espn.co.uk/rugby/results/_/team/"
        team_id = teams_list[team][1]

        team_page_resp = requests.get(
            results_base_url+team_id+'/league/{}/season/{}'.format(league, season), \
            headers=headers).content
        team_soup = BeautifulSoup(team_page_resp, 'html.parser')
        
        full_sched = team_soup.find(id='sched-container')
        match_months = full_sched.find_all('tbody')
    
    else:
        sched_dict = {}

        results_base_url = "https://www.espn.co.uk/rugby/results/_/team/"
        for team in teams_list.keys():
            team_id = teams_list[team][1]
            if team_id is not None:

                team_page_resp = requests.get(
                    results_base_url+team_id+'/league/{}/season/{}'.format(league, season),
                    headers=headers).content
                team_soup = BeautifulSoup(team_page_resp, 'html.parser')

                full_sched = team_soup.find(id='sched-container')
                match_months = full_sched.find_all('tbody')

                sched_dict[team] = match_months 

    return sched_dict


def parse_scheds(sched):
    
    totals = []
    for mon in sched:

        rows = mon.find_all('tr')

        for row in rows:

            first_row = row.find_all('td')
            date = first_row[0].text

            home_base, away_base = first_row[1].find_all('a')[0], \
                    first_row[2].find_all('a')[0]

            home_team, away_team = home_base.find('span').text, \
                    away_base.find('span').text

            home_team_abbr, away_team_abbr = home_base.find('abbr').text, \
                    away_base.find('abbr').text

            try:
                game_link = first_row[1].find_all('span')[-1].find('a')['href']
                game_id, league_id = game_link[
                    game_link.find('Id/')+3:game_link.find('/league')], \
                game_link[game_link.find('league/')+7:]
                score = first_row[1].find_all('span')[-1].find('a').text

            except TypeError:
                game_link, game_id, league_id = np.nan, np.nan, np.nan 
                score = first_row[1].find_all('span')[-1].text
    
            competition, stadium = first_row[4].text, first_row[5].text
            home_score, away_score = score.split()[0], score.split()[-1]

            totals.append([
                date, home_team, away_team, home_team_abbr,
                away_team_abbr, game_link, score, home_score,
                away_score, competition, stadium, game_id, league_id
            ])

    comb_df = pd.DataFrame(
        totals,
        columns=[
            'date', 'home_team', 'away_team', 
            'home_team_abbr', 'away_team_abbr', 'game_link', 
            'score', 'home_score', 'away_score', 
            'competition', 'stadium', 'game_id', 'league_id']
    )

    return comb_df 

In [3]:
df, nav_bar_items = get_match_stats(602502, 180659)

In [3]:
champ_cup_league = 271937
chall_cup_league = 272073
season_test_cc = 2021 

team_links_cc = get_teams_list(season_test_cc, champ_cup_league)
team_links_cc


{'Leinster': ['/rugby/team/_/id/25924/leinster', '25924'],
 'Wasps': ['/rugby/team/_/id/25905/wasps', '25905'],
 'Bordeaux Begles': ['/rugby/team/_/id/143737/bordeaux-begles', '143737'],
 'La Rochelle': ['/rugby/team/_/id/119318/la-rochelle', '119318'],
 'Scarlets': ['/rugby/team/_/id/25966/scarlets', '25966'],
 'Edinburgh': ['/rugby/team/_/id/25951/edinburgh', '25951'],
 'Toulon': ['/rugby/team/_/id/25986/toulon', '25986'],
 'Sale Sharks': ['/rugby/team/_/id/25908/sale-sharks', '25908'],
 'Northampton Saints': ['/rugby/team/_/id/25907/northampton-saints', '25907'],
 'Bath Rugby': ['/rugby/team/_/id/25898/bath-rugby', '25898'],
 'Montpellier Herault': ['/rugby/team/_/id/25918/montpellier-herault',
  '25918'],
 'Dragons': ['/rugby/team/_/id/25967/dragons', '25967'],
 'Lyon': ['/rugby/team/_/id/143736/lyon', '143736'],
 'Racing 92': ['/rugby/team/_/id/99855/racing-92', '99855'],
 'Stade Toulousain': ['/rugby/team/_/id/25922/stade-toulousain', '25922'],
 'Munster': ['/rugby/team/_/id/2592

In [5]:
len(team_links_cc)

24

In [3]:
season_test = 2021
league_test = 270557

team_links = get_teams_list(season_test, league_test)
team_sched = get_schedule(
    teams_list=team_links,
    league=league_test,
    season=season_test
)

### Gathering the Time of Game in which Red Card or Yellow card was given out 

In [2]:
# options = webdriver.ChromeOptions()
# driver = webdriver.Chrome(options=options)


In [2]:
total_club_df = pd.concat(
    [urc_game_df, t14_game_df, prem_game_df, champ_cup_game_df]
)



# total_club_df = format_score_diff_feats(total_club_df)

In [4]:
rc_games = total_club_df[total_club_df['home_red_cards'] >= 1]

In [5]:
# from bs4 import BeautifulSoup

# game_check = 598075
# league_check = 270557

# URL declaration, scraping, and initial parsing to get the tables on the page 
# url = 'https://www.espn.com/rugby/commentary/_/gameId/{}/league/{}'.format(
#     game_check, league_check
# )
# response = requests.get(url, headers=headers).content
# soup = BeautifulSoup(response, 'html.parser')

In [3]:
def grab_match_comms_cards(game, league):
    # URL declaration, scraping, and initial parsing to get the tables on the page 
    url = 'https://www.espn.com/rugby/commentary/_/gameId/{}/league/{}'.format(
        game, league
    )
    response = requests.get(url, headers=headers).content
    soup = BeautifulSoup(response, 'html.parser')

    comm_table = soup.find_all('tbody')[4]
    table_rows = [i.find_all('td') for i in comm_table.find_all('tr')]

    base_data = []
    for row in table_rows:
        row_dict = {
            'game_id': game,
            'league_id': league,
            'Time': None,
            'Team': None,
            'Player': None,
            'Event_Type': None,
            'Home_Score': None, 
            'Away_Score': None
        }

        if '+' in row[0].text:
            time_base = int(row[0].text.split("'")[0])
            time_end = int(row[0].text.split('+')[1])
            row_dict['Time'] = time_base + time_end
        else:
            row_dict['Time'] = int(row[0].text[:-1])

        event_desc = row[-1].text
        if 'Red card' in event_desc:
            row_dict['Event_Type'] = 'Red_Card'
            row_dict['Player'] = event_desc[event_desc.find('card -')+7:event_desc.find(',')-1]

        elif 'Yellow card' in event_desc:
            row_dict['Event_Type'] = 'Yellow_Card'
            row_dict['Player'] = event_desc[event_desc.find('card -')+7:event_desc.find(',')-1]

        elif 'Conversion -' in event_desc:
            row_dict['Event_Type'] = 'Conversion'
            row_dict['Player'] = event_desc[event_desc.find('Conversion -')+13:event_desc.find(',')-1]

        elif 'Try -' in event_desc:
            row_dict['Event_Type'] = 'Try'
            row_dict['Player'] = event_desc[event_desc.find('Try -')+6:event_desc.find(',')-1]

        elif 'Penalty goal -' in event_desc:
            row_dict['Event_Type'] = 'Penalty_Goal'
            row_dict['Player'] = event_desc[event_desc.find('goal -')+7:event_desc.find(',')-1]

        elif 'Substitute on' in event_desc:
            row_dict['Event_Type'] = 'Sub_In'
            row_dict['Player'] = event_desc[event_desc.find('on -')+5:event_desc.find(',')-1]

        elif 'Player substituted' in event_desc:
            row_dict['Event_Type'] = 'Sub_Out'
            row_dict['Player'] = event_desc[event_desc.find('tuted -')+8:event_desc.find(',')-1]
        
        else:
            # print('here')
            continue
            # print('now_here')

        if row_dict['Event_Type'] in ['Try', 'Penalty_Goal', 'Conversion']:
            scores = event_desc.split(' ')[0].split('-')
            row_dict['Home_Score'] = int(scores[0])
            row_dict['Away_Score'] = int(scores[1])
        
        row_dict['Team'] = event_desc[event_desc.find(',')+2:]
        base_data.append(row_dict)
    
    df = pd.DataFrame(base_data)
    df['Home_Score'] = df['Home_Score'].bfill()
    df['Away_Score'] = df['Away_Score'].bfill()

    return df




In [10]:
game = 191369
league = 270557

# URL declaration, scraping, and initial parsing to get the tables on the page 
url = 'https://www.espn.com/rugby/commentary/_/gameId/{}/league/{}'.format(
    game, league
)
response = requests.get(url, headers=headers).content
soup = BeautifulSoup(response, 'html.parser')

comm_table = soup.find_all('tbody')[4]
table_rows = [i.find_all('td') for i in comm_table.find_all('tr')]


In [12]:
table_rows[0][-1].text

'12-17 End of second half'

In [16]:
grab_match_comms_cards(191369, 270557)

here
here
here
here


,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score
0,191369,270557,79,Glasgow Warriors,Mark Bennett,Sub_Out,12.0,17.0
1,191369,270557,79,Glasgow Warriors,Scott Wight,Sub_In,12.0,17.0
2,191369,270557,79,Scarlets,Adam Warren,Sub_In,12.0,17.0
3,191369,270557,79,Scarlets,Gareth Maule,Sub_Out,12.0,17.0
4,191369,270557,76,Scarlets,Aled Davies,Sub_In,12.0,17.0
5,191369,270557,76,Scarlets,Rhodri Williams,Sub_Out,12.0,17.0
6,191369,270557,75,Glasgow Warriors,Nikola Matawalu,Yellow_Card,12.0,17.0
7,191369,270557,75,Scarlets,Jacobie Adriaanse,Sub_Out,12.0,17.0
8,191369,270557,75,Scarlets,Rhodri Jones,Sub_In,12.0,17.0
9,191369,270557,73,Scarlets,Rhodri Jones,Sub_Out,12.0,17.0


In [4]:
 
input_df = total_club_df[['game_id', 'league_id', 'season']]

league_ids = [270557, 267979, 270559, 271937]
league_names = ['URC', 'Prem', 'T14', 'ChampCup']
league_index = 1



error_games = []

filt_df = input_df[input_df['league_id'] == league_ids[league_index]].reset_index(drop=True)
league_name = league_names[league_index]

base_dfs = []
for i in tqdm(range(len(filt_df))):
    try:
        game_df = grab_match_comms_cards(filt_df['game_id'].iloc[i], filt_df['league_id'].iloc[i])
        base_dfs.append(game_df)
    except: 
        error_games.append(filt_df['game_id'].iloc[i])
        continue

all_df = pd.concat(base_dfs, axis=0)
print("Missed games amount: {}".format(len(error_games)))

100%|██████████| 1741/1741 [14:07<00:00,  2.05it/s]

Missed games amount: 16


In [5]:
total_club_df[total_club_df['game_id'].isin(error_games)][['game_id', 'league_id', 'season']]

,game_id,league_id,season
20,593898,267979,2021
72,593956,267979,2022
73,593950,267979,2022
88,593959,267979,2022
110,593964,267979,2022
111,593958,267979,2022
112,593952,267979,2022
121,593957,267979,2022
122,593953,267979,2022
129,593955,267979,2022


In [12]:
total_club_df[(total_club_df['game_id'].isin(error_games)) & (total_club_df['season'] == 2022)]

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent,date_formatted,league_season
24,598964,271937,Racing 92,13,La Rochelle,20,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-05-15,2021-2022
25,598962,271937,Racing 92,41,Sale Sharks,22,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-05-08,2021-2022
26,598958,271937,Racing 92,33,Stade Francais Paris,22,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-04-17,2021-2022
27,598947,271937,Stade Francais Paris,9,Racing 92,22,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-04-09,2021-2022
28,598940,271937,Racing 92,28,Northampton Saints,0,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-01-23,2021-2022
29,598922,271937,Ospreys,10,Racing 92,25,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-01-15,2021-2022
30,598957,271937,Ulster,23,Stade Toulousain,30,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-04-16,2021-2022
31,598946,271937,Stade Toulousain,20,Ulster,26,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-04-09,2021-2022
32,598935,271937,Ulster,34,Clermont Auvergne,31,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-01-22,2021-2022
33,598928,271937,Northampton Saints,20,Ulster,24,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-01-16,2021-2022


In [6]:
error_games

[np.int64(598907),
 np.int64(598895),
 np.int64(598917),
 np.int64(598905),
 np.int64(598915),
 np.int64(598896),
 np.int64(598909),
 np.int64(598904),
 np.int64(598900),
 np.int64(598911),
 np.int64(598902),
 np.int64(598908),
 np.int64(598890),
 np.int64(598901),
 np.int64(598913),
 np.int64(598893),
 np.int64(598914),
 np.int64(598898),
 np.int64(598918),
 np.int64(598912),
 np.int64(598916),
 np.int64(598903),
 np.int64(598910),
 np.int64(598906),
 np.int64(598964),
 np.int64(598962),
 np.int64(598958),
 np.int64(598947),
 np.int64(598940),
 np.int64(598922),
 np.int64(598957),
 np.int64(598946),
 np.int64(598935),
 np.int64(598928),
 np.int64(598965),
 np.int64(598963),
 np.int64(598960),
 np.int64(598951),
 np.int64(598943),
 np.int64(598932),
 np.int64(598927),
 np.int64(598961),
 np.int64(598954),
 np.int64(598944),
 np.int64(598937),
 np.int64(598937),
 np.int64(598924),
 np.int64(598955),
 np.int64(598948),
 np.int64(598942),
 np.int64(598925),
 np.int64(598925),
 np.int64(59

In [ ]:
filt_df

In [7]:
total_club_df[total_club_df['game_id'].isin(error_games)]

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent,date_formatted,league_season


In [5]:
error_games

[356,
 357,
 358,
 359,
 360,
 361,
 362,
 363,
 364,
 365,
 366,
 367,
 368,
 369,
 370,
 371,
 372,
 373,
 374,
 375,
 376,
 377,
 378,
 379,
 380,
 381,
 382,
 383,
 384,
 385,
 386,
 387,
 388,
 389,
 390,
 391,
 392,
 393,
 394,
 395,
 396,
 397,
 398,
 399,
 400,
 401,
 402,
 403,
 404,
 405,
 406,
 407,
 408,
 409,
 410,
 411,
 412,
 413,
 414,
 415,
 416,
 417,
 418,
 419,
 420,
 421,
 422,
 423,
 424,
 425,
 426,
 427,
 428,
 429,
 430,
 432,
 434,
 435,
 436,
 437,
 440,
 441,
 442,
 443,
 444,
 445,
 447,
 448,
 450,
 451,
 452,
 453,
 455,
 456,
 457,
 458,
 459,
 460,
 461,
 462,
 463,
 464,
 465,
 466,
 467,
 468,
 469,
 470,
 471,
 472,
 473,
 474,
 475,
 476,
 477,
 478,
 479,
 480,
 481,
 482,
 483,
 484,
 485,
 486,
 487,
 488,
 489,
 490,
 491,
 492,
 493,
 494,
 495]

In [14]:
all_df[all_df['game_id'] == all_df['game_id'].iloc[0]].drop_duplicates()

,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score
0,597160,271937,82,Sale Sharks,Robert du Preez,Conversion,37.0,27.0
1,597160,271937,81,Sale Sharks,Tom Curtis,Try,37.0,25.0
2,597160,271937,80,Leinster,Charlie Ngatai,Sub_Out,37.0,20.0
3,597160,271937,80,Sale Sharks,Robert du Preez,Conversion,37.0,20.0
4,597160,271937,79,Sale Sharks,Tommy Taylor,Try,37.0,18.0
5,597160,271937,78,Leinster,Hugo Keenan,Yellow_Card,37.0,13.0
6,597160,271937,71,Leinster,Sam Prendergast,Conversion,37.0,13.0
7,597160,271937,70,Leinster,Cian Healy,Try,35.0,13.0
8,597160,271937,70,Leinster,Ben Murphy,Sub_In,30.0,13.0
9,597160,271937,70,Leinster,Jamison Gibson-Park,Sub_Out,30.0,13.0


In [ ]:
output_df = all_df[~all_df['Team'].str.contains(' half')].drop_.reset_index()
output_df

,index,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score
0,0,597814,270559,82,Stade Toulousain,Clement Verge,Try,22.0,8.0
1,1,597814,270559,76,Stade Toulousain,Juan Cruz Mallia,Yellow_Card,22.0,3.0
2,2,597814,270559,70,La Rochelle,Judicael Cancoriet,Sub_In,22.0,3.0
3,3,597814,270559,70,La Rochelle,Levani Botia,Sub_Out,22.0,3.0
4,4,597814,270559,59,La Rochelle,Ultan Dillane,Sub_In,22.0,3.0
...,...,...,...,...,...,...,...,...,...
95524,44,268769,270559,23,Agen,Alexi Bales,Try,3.0,11.0
95525,45,268769,270559,20,Agen,Burton Francis,Penalty_Goal,3.0,6.0
95526,46,268769,270559,20,Oyonnax,Jody Jenneker,Yellow_Card,3.0,3.0
95527,47,268769,270559,16,Agen,Burton Francis,Penalty_Goal,3.0,3.0


In [6]:
all_df.drop_duplicates().to_csv('formed_data/match_comm_data/Prem_data_v2.csv', index=False)

In [20]:
pd.concat(base_dfs[:3], axis=0)

,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score
0,167053,270557,80,7-0 End of second half,None,None,17.0,0.0
1,167053,270557,78,Leinster,John Cooney,Sub_In,17.0,0.0
2,167053,270557,78,Leinster,Ian Madigan,Sub_Out,17.0,0.0
3,167053,270557,75,Leinster,Brendan Macken,Sub_Out,17.0,0.0
4,167053,270557,75,Leinster,Ian Madigan,Conversion,17.0,0.0
...,...,...,...,...,...,...,...,...
44,167039,270557,32,Zebre,Quintin Geldenhuys,Yellow_Card,10.0,0.0
45,167039,270557,28,Leinster,Johnny Sexton,Conversion,10.0,0.0
46,167039,270557,26,Leinster,Andrew Conway,Try,8.0,0.0
47,167039,270557,25,Leinster,Johnny Sexton,Penalty_Goal,3.0,0.0


In [14]:
base_dfs[-3]

,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score
0,290555,270557,81,-19 End of second half,None,None,3.0,19.0
1,290555,270557,78,Benetton Treviso,Federico Zani,Sub_Out,3.0,19.0
2,290555,270557,78,Benetton Treviso,Alberto Porolli,Sub_In,3.0,19.0
3,290555,270557,76,Benetton Treviso,Francesco Minto,Sub_Out,3.0,19.0
4,290555,270557,76,Benetton Treviso,Simone Ferrari,Sub_In,3.0,19.0
5,290555,270557,76,Benetton Treviso,David Odiete,Sub_Out,3.0,19.0
6,290555,270557,76,Benetton Treviso,Tommaso Allan,Sub_In,3.0,19.0
7,290555,270557,75,Zebre,Dries van Schalkwyk,Sub_Out,3.0,19.0
8,290555,270557,75,Zebre,Kayle van Zyl,Sub_In,3.0,19.0
9,290555,270557,75,Benetton Treviso,Tiziano Pasquali,Yellow_Card,3.0,19.0


In [13]:
base_dfs[-2]

,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score
0,290496,270557,81,6-8 End of second half,None,None,26.0,8.0
1,290496,270557,75,Benetton Treviso,Tommaso Iannone,Sub_In,26.0,8.0
2,290496,270557,75,Benetton Treviso,Braam Steyn,Sub_Out,26.0,8.0
3,290496,270557,69,Dragons,Dorian Jones,Conversion,26.0,8.0
4,290496,270557,69,Dragons,Lewis Evans,Try,24.0,8.0
5,290496,270557,64,Dragons,Brok Harris,Sub_Out,19.0,8.0
6,290496,270557,64,Dragons,Lloyd Fairbrother,Sub_In,19.0,8.0
7,290496,270557,63,Dragons,Elliot Dee,Sub_Out,19.0,8.0
8,290496,270557,63,Dragons,Rhys Buckley,Sub_In,19.0,8.0
9,290496,270557,63,Dragons,Angus O'Brien,Sub_Out,19.0,8.0


In [12]:
base_dfs[-1]

,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score
0,290537,270557,81,9-14 End of second half,None,None,29.0,14.0
1,290537,270557,78,Dragons,Adam Warren,Sub_In,29.0,14.0
2,290537,270557,78,Dragons,Tyler Morgan,Sub_Out,29.0,14.0
3,290537,270557,78,Dragons,Darran Harris,Sub_In,29.0,14.0
4,290537,270557,78,Dragons,Rhys Buckley,Sub_Out,29.0,14.0
5,290537,270557,69,Zebre,Serafin Bordoli,Yellow_Card,29.0,14.0
6,290537,270557,69,Zebre,Guglielmo Palazzani,Sub_In,29.0,14.0
7,290537,270557,69,Zebre,Kayle van Zyl,Sub_Out,29.0,14.0
8,290537,270557,67,Dragons,Tom Prydie,Sub_In,29.0,14.0
9,290537,270557,67,Dragons,Pat Howard,Sub_Out,29.0,14.0


In [10]:
all_df

,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score,game_id,league_id,...,Home_Score,Away_Score,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score
0,167053.0,270557.0,80.0,7-0 End of second half,None,None,17.0,0.0,167046.0,270557.0,...,26.0,8.0,290537.0,270557.0,81.0,9-14 End of second half,None,None,29.0,14.0
1,167053.0,270557.0,78.0,Leinster,John Cooney,Sub_In,17.0,0.0,167046.0,270557.0,...,26.0,8.0,290537.0,270557.0,78.0,Dragons,Adam Warren,Sub_In,29.0,14.0
2,167053.0,270557.0,78.0,Leinster,Ian Madigan,Sub_Out,17.0,0.0,167046.0,270557.0,...,26.0,8.0,290537.0,270557.0,78.0,Dragons,Tyler Morgan,Sub_Out,29.0,14.0
3,167053.0,270557.0,75.0,Leinster,Brendan Macken,Sub_Out,17.0,0.0,167046.0,270557.0,...,26.0,8.0,290537.0,270557.0,78.0,Dragons,Darran Harris,Sub_In,29.0,14.0
4,167053.0,270557.0,75.0,Leinster,Ian Madigan,Conversion,17.0,0.0,167046.0,270557.0,...,24.0,8.0,290537.0,270557.0,78.0,Dragons,Rhys Buckley,Sub_Out,29.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [116]:
test_df = grab_match_comms_cards(game=598075, league=270557)

In [117]:
test_df

,game_id,league_id,Time,Team,Player,Event_Type,Home_Score,Away_Score
0,598075,270557,75,Glasgow Warriors,George Horne,Penalty_Goal,10.0,17.0
1,598075,270557,74,Munster,Alex Nankivell,Red_Card,10.0,14.0
2,598075,270557,73,Munster,Alex Kendellen,Sub_In,10.0,14.0
3,598075,270557,73,Munster,John Hodnett,Sub_Out,10.0,14.0
4,598075,270557,71,Glasgow Warriors,Matt Fagerson,Sub_In,10.0,14.0
5,598075,270557,71,Glasgow Warriors,Euan Ferrie,Sub_Out,10.0,14.0
6,598075,270557,71,Glasgow Warriors,Oli Kebble,Sub_In,10.0,14.0
7,598075,270557,71,Glasgow Warriors,Jamie Bhatti,Sub_Out,10.0,14.0
8,598075,270557,66,Munster,John Ryan,Sub_In,10.0,14.0
9,598075,270557,66,Munster,Jeremy Loughman,Sub_Out,10.0,14.0


### Continuing to Other Maintenance Tasks 

In [2]:
def assign_season(df):
    if df['month'] >= 9:
        league_season = str((df['season'])) + '-' + str(df['season'] + 1)
    elif df['league_id'] == 242041:
        league_season = df['season']
    else:
        league_season = str(df['season'] - 1) + '-' + str(df['season'])
    return league_season
    

def rejoin_date_df(sched_df, game_df,path):
    # sched_df[['month', 'day']] = sched_df['date'].apply(parse_game_date_v2).apply(pd.Series)
    # sched_df['date_formatted'] = sched_df.apply(lambda x: datetime.datetime(x['season'], x['month'], x['day']), axis=1)

    sched_df['date_formatted'] = sched_df.apply(parse_game_date_v3, axis=1)
    sched_df[['month', 'day']] = sched_df['date_formatted'].apply(lambda x: (x.month, x.day)).apply(pd.Series)
    sched_df['league_season'] = sched_df.apply(assign_season, axis=1)

    merged_df = game_df.merge(
        sched_df[['game_id', 'date_formatted', 'league_season']],
        how='left',
        on='game_id'
    )

    grouped_df = merged_df.groupby('league_season')
    season_dict = {name: group for name, group in grouped_df}

    for season, value in season_dict.items():
        value.to_csv(path+'_'+str(season)+'.csv', index=False)
        print("DF for season {} saved to {}".format(str(season), path+'_'+str(season)+'.csv'))


def run_joiner_output(sched_path, game_path, output_path):
    sched_concat_df = pd.concat(list(map(pd.read_csv, sched_path)))
    game_concat_df = pd.concat(list(map(pd.read_csv, game_path)))

    sched_concat_df.loc[sched_concat_df['date'] == 'Mar, Dic 26', 'date'] = 'Tue, Dec 26'

    rejoin_date_df(
        sched_df=sched_concat_df, 
        game_df=game_concat_df, 
        path=output_path
    )

    

In [8]:
sched_files_use_rchamp = [sched_path + '/' + f'RChamp_schedule_data_{i}.csv' for i in range(2011, 2024) if i != 2020]
game_files_use_rchamp = [game_path + '/' + f'RChamp_game_stats_{i}.csv' for i in range(2011, 2024) if i != 2020]

sched_files_use_prem = [sched_path + '/' + f'Prem_schedule_data_{i}.csv' for i in range(2013, 2019) if i != 2020]
game_files_use_prem = [game_path + '/' + f'Prem_game_stats_{i}.csv' for i in range(2013, 2019) if i != 2020]


In [12]:
sched_files_use_champ_cup = [sched_path + '/' + f'ChampCup_schedule_data_{i}.csv' for i in range(2011, 2024)]
game_files_use_champ_cup = [game_path + '/' + f'ChampCup_game_stats_{i}.csv' for i in range(2011, 2024)]

champ_cup_file_path = 'formed_data/game_data_v2/ChampCup_game_data_v2'
run_joiner_output(sched_files_use_champ_cup, game_files_use_champ_cup, champ_cup_file_path)


DF for season 2010-2011 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2010-2011.csv
DF for season 2011-2012 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2011-2012.csv
DF for season 2012-2013 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2012-2013.csv
DF for season 2013-2014 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2013-2014.csv
DF for season 2014-2015 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2014-2015.csv
DF for season 2015-2016 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2015-2016.csv
DF for season 2016-2017 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2016-2017.csv
DF for season 2017-2018 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2017-2018.csv
DF for season 2018-2019 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2018-2019.csv
DF for season 2019-2020 saved to formed_data/game_data_v2/ChampCup_game_data_v2_2019-2020.csv
DF for season 2020-2021 saved to formed_data/game_data_v2/Ch

In [6]:
sched_files_use_urc = [sched_path + '/' + f'URC_schedule_data_{i}.csv' for i in range(2011, 2024)]
game_files_use_urc = [game_path + '/' + f'URC_game_stats_{i}.csv' for i in range(2011, 2024)]

urc_file_path = 'formed_data/game_data_v2/URC_game_data_v2'
run_joiner_output(sched_files_use_urc, game_files_use_urc, urc_file_path)


DF for season 2010-2011 saved to formed_data/game_data_v2/URC_game_data_v2_2010-2011.csv
DF for season 2011-2012 saved to formed_data/game_data_v2/URC_game_data_v2_2011-2012.csv
DF for season 2012-2013 saved to formed_data/game_data_v2/URC_game_data_v2_2012-2013.csv
DF for season 2013-2014 saved to formed_data/game_data_v2/URC_game_data_v2_2013-2014.csv
DF for season 2014-2015 saved to formed_data/game_data_v2/URC_game_data_v2_2014-2015.csv
DF for season 2015-2016 saved to formed_data/game_data_v2/URC_game_data_v2_2015-2016.csv
DF for season 2016-2017 saved to formed_data/game_data_v2/URC_game_data_v2_2016-2017.csv
DF for season 2017-2018 saved to formed_data/game_data_v2/URC_game_data_v2_2017-2018.csv
DF for season 2018-2019 saved to formed_data/game_data_v2/URC_game_data_v2_2018-2019.csv
DF for season 2019-2020 saved to formed_data/game_data_v2/URC_game_data_v2_2019-2020.csv
DF for season 2020-2021 saved to formed_data/game_data_v2/URC_game_data_v2_2020-2021.csv
DF for season 2021-20

In [5]:
sched_files_use_prem = [sched_path + '/' + f'Prem_schedule_data_{i}.csv' for i in range(2011, 2024)]
game_files_use_prem = [game_path + '/' + f'Prem_game_stats_{i}.csv' for i in range(2011, 2024)]

prem_file_path = 'formed_data/game_data_v2/Prem_game_data_v2'
run_joiner_output(sched_files_use_prem, game_files_use_prem, prem_file_path)

DF for season 2010-2011 saved to formed_data/game_data_v2/Prem_game_data_v2_2010-2011.csv
DF for season 2011-2012 saved to formed_data/game_data_v2/Prem_game_data_v2_2011-2012.csv
DF for season 2012-2013 saved to formed_data/game_data_v2/Prem_game_data_v2_2012-2013.csv
DF for season 2013-2014 saved to formed_data/game_data_v2/Prem_game_data_v2_2013-2014.csv
DF for season 2014-2015 saved to formed_data/game_data_v2/Prem_game_data_v2_2014-2015.csv
DF for season 2015-2016 saved to formed_data/game_data_v2/Prem_game_data_v2_2015-2016.csv
DF for season 2016-2017 saved to formed_data/game_data_v2/Prem_game_data_v2_2016-2017.csv
DF for season 2017-2018 saved to formed_data/game_data_v2/Prem_game_data_v2_2017-2018.csv
DF for season 2018-2019 saved to formed_data/game_data_v2/Prem_game_data_v2_2018-2019.csv
DF for season 2019-2020 saved to formed_data/game_data_v2/Prem_game_data_v2_2019-2020.csv
DF for season 2020-2021 saved to formed_data/game_data_v2/Prem_game_data_v2_2020-2021.csv
DF for sea

In [7]:
sched_files_use_sr = [sched_path + '/' + f'SR_schedule_data_{i}.csv' for i in range(2011, 2021)]
game_files_use_sr = [game_path + '/' + f'SR_game_stats_{i}.csv' for i in range(2011, 2021)]

sr_file_path = 'formed_data/game_data_v2/SR_game_data_v2'
run_joiner_output(sched_files_use_sr, game_files_use_sr, sr_file_path)


DF for season 2011 saved to formed_data/game_data_v2/SR_game_data_v2_2011.csv
DF for season 2012 saved to formed_data/game_data_v2/SR_game_data_v2_2012.csv
DF for season 2013 saved to formed_data/game_data_v2/SR_game_data_v2_2013.csv
DF for season 2014 saved to formed_data/game_data_v2/SR_game_data_v2_2014.csv
DF for season 2015 saved to formed_data/game_data_v2/SR_game_data_v2_2015.csv
DF for season 2016 saved to formed_data/game_data_v2/SR_game_data_v2_2016.csv
DF for season 2017 saved to formed_data/game_data_v2/SR_game_data_v2_2017.csv
DF for season 2018 saved to formed_data/game_data_v2/SR_game_data_v2_2018.csv
DF for season 2019 saved to formed_data/game_data_v2/SR_game_data_v2_2019.csv
DF for season 2020 saved to formed_data/game_data_v2/SR_game_data_v2_2020.csv


In [6]:
sched_files_use_sr

['formed_data/game_schedule_data/SR_schedule_data_2011.csv',
 'formed_data/game_schedule_data/SR_schedule_data_2012.csv',
 'formed_data/game_schedule_data/SR_schedule_data_2013.csv',
 'formed_data/game_schedule_data/SR_schedule_data_2014.csv',
 'formed_data/game_schedule_data/SR_schedule_data_2015.csv',
 'formed_data/game_schedule_data/SR_schedule_data_2016.csv',
 'formed_data/game_schedule_data/SR_schedule_data_2017.csv',
 'formed_data/game_schedule_data/SR_schedule_data_2018.csv',
 'formed_data/game_schedule_data/SR_schedule_data_2019.csv',
 'formed_data/game_schedule_data/SR_schedule_data_2020.csv']

In [ ]:
six_nations_file_path = 'formed_data/game_data_v2/SixNat_game_data_v2'
rchamp_file_path = 'formed_data/game_data_v2/RChamp_game_data_v2'


run_joiner_output(six_nations_sched_files, six_nations_game_files, six_nations_file_path)
run_joiner_output(sched_files_use_rchamp, game_files_use_rchamp, rchamp_file_path)


DF for season 2010-2011 saved to formed_data/game_data_v2/SixNat_game_data_v2_2010-2011.csv
DF for season 2011-2012 saved to formed_data/game_data_v2/SixNat_game_data_v2_2011-2012.csv
DF for season 2012-2013 saved to formed_data/game_data_v2/SixNat_game_data_v2_2012-2013.csv
DF for season 2013-2014 saved to formed_data/game_data_v2/SixNat_game_data_v2_2013-2014.csv
DF for season 2014-2015 saved to formed_data/game_data_v2/SixNat_game_data_v2_2014-2015.csv
DF for season 2015-2016 saved to formed_data/game_data_v2/SixNat_game_data_v2_2015-2016.csv
DF for season 2016-2017 saved to formed_data/game_data_v2/SixNat_game_data_v2_2016-2017.csv
DF for season 2017-2018 saved to formed_data/game_data_v2/SixNat_game_data_v2_2017-2018.csv
DF for season 2018-2019 saved to formed_data/game_data_v2/SixNat_game_data_v2_2018-2019.csv
DF for season 2019-2020 saved to formed_data/game_data_v2/SixNat_game_data_v2_2019-2020.csv
DF for season 2020-2021 saved to formed_data/game_data_v2/SixNat_game_data_v2_20

In [ ]:
sched_concat_t14_df = pd.concat(list(map(pd.read_csv, sched_files_use_t14)))
game_concat_t14_df = pd.concat(list(map(pd.read_csv, game_files_use_t14)))

In [ ]:
sched_concat_t14_df = pd.concat(list(map(pd.read_csv, sched_files_use_s)))
game_concat_t14_df = pd.concat(list(map(pd.read_csv, game_files_use_t14)))

In [34]:
sched_files_use_urc = [sched_path + '/' + f'URC_schedule_data_{i}.csv' for i in range(2011, 2020)]
game_files_use_urc= [game_path + '/' + f'URC_game_stats_{i}.csv' for i in range(2011, 2020)]

sched_files_use_t14 = [sched_path + '/' + f'T14_schedule_data_{i}.csv' for i in range(2011, 2020) if i != 2013]
# sched_files_use_t14 = sched_files_use_t14.remove('formed_data/game_schedule_data/T14_schedule_data_2014.csv')
game_files_use_t14 = [game_path + '/' + f'T14_game_stats_{i}.csv' for i in range(2011, 2020) if i != 2013]
# game_files_use_t14 = game_files_use_t14.remove('formed_data/game_data/T14_game_stats_2014.csv')
sched_files_use_t14

['formed_data/game_schedule_data/T14_schedule_data_2011.csv',
 'formed_data/game_schedule_data/T14_schedule_data_2012.csv',
 'formed_data/game_schedule_data/T14_schedule_data_2014.csv',
 'formed_data/game_schedule_data/T14_schedule_data_2015.csv',
 'formed_data/game_schedule_data/T14_schedule_data_2016.csv',
 'formed_data/game_schedule_data/T14_schedule_data_2017.csv',
 'formed_data/game_schedule_data/T14_schedule_data_2018.csv',
 'formed_data/game_schedule_data/T14_schedule_data_2019.csv']

In [9]:
sched_concat_t14_df = pd.concat(list(map(pd.read_csv, sched_files_use_t14)))
game_concat_t14_df = pd.concat(list(map(pd.read_csv, game_files_use_t14)))

In [29]:
sched_concat_df = pd.concat(list(map(pd.read_csv, sched_files_use_urc)))
game_concat_df = pd.concat(list(map(pd.read_csv, game_files_use_urc)))

In [10]:
sched_concat_t14_df['date']

0      Fri, Dec 30
1      Fri, Dec 23
2       Sat, Dec 3
3      Sat, Nov 26
4       Sat, Nov 5
          ...     
183     Sat, Oct 5
184    Sat, Aug 31
185     Sat, May 4
186    Sat, Feb 23
187    Sat, Mar 23
Name: date, Length: 1477, dtype: object

In [11]:
sched_concat_df['date']

0     Fri, Dec 30
1     Mon, Dec 26
2      Sat, Dec 3
3     Sat, Nov 26
4      Fri, Nov 4
         ...     
95    Sat, Sep 28
96    Sat, Apr 27
97    Sat, Apr 13
98    Sat, Feb 23
99    Sat, Feb 16
Name: date, Length: 1197, dtype: object

In [13]:
dateparser.parse(sched_concat_t14_df['date'].iloc[0], languages=['en'])

datetime.datetime(2025, 12, 30, 0, 0)

In [24]:
dateparser.parse('Mar, Dic 26', languages=['es'])

In [22]:
sched_concat_df['date'].iloc[65]

'Fri, Jan 7'

In [31]:
sched_concat_df.loc[sched_concat_df['date'] == 'Mar, Dic 26', 'date'] = 'Tue, Dec 26'

In [15]:
for i in range(len(sched_concat_df)):
    if dateparser.parse(sched_concat_df['date'].iloc[0], languages=['en']) == None:
        print(i)

In [12]:
sched_concat_t14_df['date_formatted'] = sched_concat_t14_df.apply(parse_game_date_v3, axis=1)


In [19]:
sched_concat_df[sched_concat_df['date'] == 'Mar, Dic 26']

,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
65,"Mar, Dic 26",Scarlets,Ospreys,SCA,OSP,/rugby/match/_/gameId/291940/league/270557,12 - 9,12,9,United Rugby Championship,"Parc y Scarlets, Llanelli",291940.0,270557.0,2017


In [26]:
sched_concat_df['date_formatted'] = sched_concat_df.apply(parse_game_date_v3, axis=1)
sched_concat_df['date_formatted']

Fri, Dec 30
Mon, Dec 26
Sat, Dec 3
Sat, Nov 26
Fri, Nov 4
Fri, Oct 28
Sat, Oct 8
Fri, Sep 30
Fri, Sep 23
Sat, Sep 17
Fri, Sep 9
Sat, Sep 3
Sat, May 28
Sat, May 14
Fri, May 6
Sat, Apr 23
Sat, Apr 16
Sat, Apr 2
Fri, Mar 25
Sat, Mar 5
Sun, Feb 27
Fri, Feb 18
Sun, Feb 13
Sat, Jan 8
Sat, Jan 1
Mon, Dec 26
Fri, Dec 2
Sat, Nov 26
Fri, Oct 28
Sat, Oct 8
Sat, Oct 1
Sat, Sep 24
Sat, Sep 17
Fri, Sep 9
Fri, Sep 2
Fri, May 13
Fri, May 6
Sat, Apr 23
Sat, Apr 16
Sun, Mar 27
Fri, Mar 4
Fri, Feb 25
Sat, Feb 19
Thu, Feb 10
Fri, Jan 7
Sat, Jan 1
Fri, Dec 2
Fri, Nov 25
Sat, Nov 5
Sat, Oct 29
Fri, Oct 7
Fri, Sep 30
Sun, Sep 25
Fri, Sep 16
Sat, Sep 10
Fri, Sep 2
Fri, May 6
Fri, Apr 22
Fri, Apr 1
Fri, Mar 25
Fri, Mar 18
Fri, Mar 4
Fri, Feb 25
Fri, Feb 18
Sun, Feb 13
Fri, Jan 7
Mon, Dec 26
Sat, Nov 26
Sat, Nov 5
Fri, Oct 28
Fri, Sep 30
Sat, Sep 17
Sat, Sep 10
Fri, May 6
Fri, Apr 15
Sat, Apr 2
Sat, Mar 26
Sat, Mar 5
Sun, Feb 27
Sat, Feb 19
Fri, Dec 30
Fri, Nov 25
Fri, Oct 7
Sat, Oct 1
Sat, Sep 10
Sun, Sep 4
Fr

0     2011-12-30
1     2011-12-26
2     2011-12-03
3     2011-11-26
4     2011-11-04
         ...    
95    2019-09-28
96    2019-04-27
97    2019-04-13
98    2019-02-23
99    2019-02-16
Name: date_formatted, Length: 1197, dtype: object

In [35]:
t14_file_path = 'formed_data/game_data_v2/T14_game_data_v2'
urc_file_path = 'formed_data/game_data_v2/URC_game_data_v2'

# run_joiner_output(sched_files_use_t14, game_files_use_t14, t14_file_path)
run_joiner_output(sched_files_use_urc, game_files_use_urc, urc_file_path)



DF for season 2010-2011 saved to formed_data/game_data_v2/URC_game_data_v2_2010-2011.csv
DF for season 2011-2012 saved to formed_data/game_data_v2/URC_game_data_v2_2011-2012.csv
DF for season 2012-2013 saved to formed_data/game_data_v2/URC_game_data_v2_2012-2013.csv
DF for season 2013-2014 saved to formed_data/game_data_v2/URC_game_data_v2_2013-2014.csv
DF for season 2014-2015 saved to formed_data/game_data_v2/URC_game_data_v2_2014-2015.csv
DF for season 2015-2016 saved to formed_data/game_data_v2/URC_game_data_v2_2015-2016.csv
DF for season 2016-2017 saved to formed_data/game_data_v2/URC_game_data_v2_2016-2017.csv
DF for season 2017-2018 saved to formed_data/game_data_v2/URC_game_data_v2_2017-2018.csv
DF for season 2018-2019 saved to formed_data/game_data_v2/URC_game_data_v2_2018-2019.csv
DF for season 2019-2020 saved to formed_data/game_data_v2/URC_game_data_v2_2019-2020.csv


In [33]:
def test_func(date_parse):
    try:
        dow = date_parse[:date_parse.find(',')]
        month = date_parse[date_parse.find(',')+2:date_parse.find(',')+5]
        month_num = datetime.datetime.strptime(month, '%b').month
        # return month_num
    except:
        print(date_parse)
        return 'error'

In [19]:
urc_sched_df = pd.concat(list(map(pd.read_csv, urc_sched_files)))

In [20]:
urc_sched_df

,Unnamed: 0,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,0,"Fri, Dec 3",Leinster,Connacht,LEI,CON,/rugby/match/_/gameId/594385/league/270557,47 - 19,47,19,United Rugby Championship,"RDS Arena, Dublin",594385.0,270557.0,2021
1,1,"Sat, Nov 27",Leinster,Ulster,LEI,ULS,/rugby/match/_/gameId/594381/league/270557,10 - 20,10,20,United Rugby Championship,"RDS Arena, Dublin",594381.0,270557.0,2021
2,2,"Fri, Oct 22",Glasgow Warriors,Leinster,GLA,LEI,/rugby/match/_/gameId/594369/league/270557,15 - 31,15,31,United Rugby Championship,"Scotstoun Stadium, Glasgow",594369.0,270557.0,2021
3,3,"Sat, Oct 16",Leinster,Scarlets,LEI,SCA,/rugby/match/_/gameId/594364/league/270557,50 - 15,50,15,United Rugby Championship,"RDS Arena, Dublin",594364.0,270557.0,2021
4,4,"Sat, Oct 9",Leinster,Zebre,LEI,ZEB,/rugby/match/_/gameId/594353/league/270557,43 - 7,43,7,United Rugby Championship,"RDS Arena, Dublin",594353.0,270557.0,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,214,"Sat, Oct 28",Ospreys,Zebre,OSP,ZEB,/rugby/match/_/gameId/597934/league/270557,34 - 31,34,31,United Rugby Championship,"Swansea.com Stadium, Swansea",597934.0,270557.0,2023
132,218,"Sat, Mar 25",Ospreys,Dragons,OSP,DRA,/rugby/match/_/gameId/599506/league/270557,37 - 18,37,18,United Rugby Championship,"Swansea.com Stadium, Swansea",599506.0,270557.0,2023
133,221,"Sun, Jan 29",Zebre,Ospreys,ZEB,OSP,/rugby/match/_/gameId/599483/league/270557,24 - 28,24,28,United Rugby Championship,"Stadio Sergio Lanfranchi, Parma",599483.0,270557.0,2023
134,232,"Sat, Apr 22",Dragons,Scarlets,DRA,SCA,/rugby/match/_/gameId/599523/league/270557,31 - 14,31,14,United Rugby Championship,"Principality Stadium, Cardiff",599523.0,270557.0,2023


In [27]:
t14_sched_df['date'].isnull().sum()

np.int64(0)

In [34]:
import dateparser

ModuleNotFoundError: No module named 'dateparser'

In [32]:
t14_sched_df

,Unnamed: 0,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,0,"Sat, Dec 4",Bordeaux Begles,Stade Toulousain,UNI,STA,/rugby/match/_/gameId/594117/league/270559,17 - 7,17,7,French Top 14,"Stade Chaban-Delmas, Bordeaux",594117.0,270559.0,2021
1,1,"Sat, Nov 27",Stade Toulousain,Brive,STA,BRIV,/rugby/match/_/gameId/594114/league/270559,18 - 11,18,11,French Top 14,"Stade Ernest-Wallon, Toulouse",594114.0,270559.0,2021
2,2,"Sat, Nov 6",Stade Toulousain,Perpignan,STA,USA,/rugby/match/_/gameId/594108/league/270559,37 - 15,37,15,French Top 14,"Stade Ernest-Wallon, Toulouse",594108.0,270559.0,2021
3,3,"Sun, Oct 31",Racing 92,Stade Toulousain,RAC,STA,/rugby/match/_/gameId/594098/league/270559,27 - 18,27,18,French Top 14,"Paris La Defense Arena, Nanterre",594098.0,270559.0,2021
4,4,"Sat, Oct 23",Stade Toulousain,Castres Olympique,STA,CAS,/rugby/match/_/gameId/594094/league/270559,41 - 0,41,0,French Top 14,"Stade Ernest-Wallon, Toulouse",594094.0,270559.0,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,308,"Sat, Oct 5",Agen,Bayonne,AGN,BAY,NaN,27 - 29,27,29,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
184,312,"Sat, Aug 31",Agen,Brive,AGN,BRIV,NaN,16 - 10,16,10,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
185,316,"Sat, May 4",Grenoble,Agen,GRENO,AGN,/rugby/match/_/gameId/293211/league/270559,11 - 29,11,29,French Top 14,"Stade des Alpes, Grenoble",293211.0,270559.0,2019
186,323,"Sat, Feb 23",Perpignan,Agen,USA,AGN,/rugby/match/_/gameId/293157/league/270559,13 - 20,13,20,French Top 14,"Stade Aime Giral, Perpignan",293157.0,270559.0,2019


In [31]:
for i in range(len(t14_sched_df)):
    test_func(t14_sched_df['date'].iloc[i])
    

Jue, Abr 29
Vie, Abr 16
Vie, Ene 22
Sáb, Ene 16
Sáb, Dic 31
Vie, Dic 23
Sáb, Dic 3
Sáb, Abr 30
Sáb, Abr 23
Sáb, Abr 2
Sáb, Ene 29
Sáb, Ene 8
Sáb, Ene 1


In [13]:
tester = t14_sched_df['date'].iloc[0]
tester

'Sat, Dec 4'

In [21]:
tester[tester.find(',')+2:-6]

''

In [22]:
tester

'Sat, Dec 4'

In [23]:
tester[:tester.find('.')]

'Sat, Dec '

In [24]:
tester[tester.find(',')+2:tester.find(',')+5]

'Dec'

In [25]:
datetime.datetime.strptime(tester[tester.find(',')+2:tester.find(',')+5], '%b').month

12

In [26]:
tester[tester.find(tester[tester.find(',')+2:tester.find(',')+5])+4:]

'4'

In [ ]:
dow = date_parse[:date_parse.find(',')]
month = date_parse[date_parse.find(',')+2:date_parse.find(',')+5]
month_num = datetime.datetime.strptime(month, '%b').month
dom = date_parse[date_parse.find(month)+4:]

In [12]:
t14_sched_df = pd.concat(list(map(pd.read_csv, t14_sched_files)))
t14_sched_df

,Unnamed: 0,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,0,"Sat, Dec 4",Bordeaux Begles,Stade Toulousain,UNI,STA,/rugby/match/_/gameId/594117/league/270559,17 - 7,17,7,French Top 14,"Stade Chaban-Delmas, Bordeaux",594117.0,270559.0,2021
1,1,"Sat, Nov 27",Stade Toulousain,Brive,STA,BRIV,/rugby/match/_/gameId/594114/league/270559,18 - 11,18,11,French Top 14,"Stade Ernest-Wallon, Toulouse",594114.0,270559.0,2021
2,2,"Sat, Nov 6",Stade Toulousain,Perpignan,STA,USA,/rugby/match/_/gameId/594108/league/270559,37 - 15,37,15,French Top 14,"Stade Ernest-Wallon, Toulouse",594108.0,270559.0,2021
3,3,"Sun, Oct 31",Racing 92,Stade Toulousain,RAC,STA,/rugby/match/_/gameId/594098/league/270559,27 - 18,27,18,French Top 14,"Paris La Defense Arena, Nanterre",594098.0,270559.0,2021
4,4,"Sat, Oct 23",Stade Toulousain,Castres Olympique,STA,CAS,/rugby/match/_/gameId/594094/league/270559,41 - 0,41,0,French Top 14,"Stade Ernest-Wallon, Toulouse",594094.0,270559.0,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,308,"Sat, Oct 5",Agen,Bayonne,AGN,BAY,NaN,27 - 29,27,29,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
184,312,"Sat, Aug 31",Agen,Brive,AGN,BRIV,NaN,16 - 10,16,10,French Top 14,"Alfred Armandie, Agen",NaN,NaN,2019
185,316,"Sat, May 4",Grenoble,Agen,GRENO,AGN,/rugby/match/_/gameId/293211/league/270559,11 - 29,11,29,French Top 14,"Stade des Alpes, Grenoble",293211.0,270559.0,2019
186,323,"Sat, Feb 23",Perpignan,Agen,USA,AGN,/rugby/match/_/gameId/293157/league/270559,13 - 20,13,20,French Top 14,"Stade Aime Giral, Perpignan",293157.0,270559.0,2019


In [10]:
urc_file_path = 'formed_data/game_data_v2/URC_game_data_v2'
prem_file_path = 'formed_data/game_data_v2/Prem_game_data_v2'
sr_file_path = 'formed_data/game_data_v2/SR_game_data_v2'

# run_joiner_output(urc_sched_files, urc_game_files, urc_file_path)
# run_joiner_output(prem_sched_files, prem_game_files, prem_file_path)
run_joiner_output(sr_sched_files, sr_game_files, sr_file_path)


DF for season 2020 saved to formed_data/game_data_v2/SR_game_data_v2_2020.csv
DF for season 2022 saved to formed_data/game_data_v2/SR_game_data_v2_2022.csv
DF for season 2023 saved to formed_data/game_data_v2/SR_game_data_v2_2023.csv
DF for season 2024 saved to formed_data/game_data_v2/SR_game_data_v2_2024.csv


In [4]:
df_rev = pd.read_csv('formed_data/game_data_v2/URC_game_data_v2_2019-2020.csv')
df_rev.head()

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent,date_formatted,league_season
0,592889,270557,Ulster,10,Leinster,28,1.0,3.0,1.0,2.0,...,7.0,1.0,6.0,6.0,1.00,14.0,14.0,1.00,2020-08-29,2019-2020
1,592884,270557,Leinster,27,Munster,25,3.0,3.0,3.0,2.0,...,10.0,0.9,4.0,4.0,1.00,13.0,13.0,1.00,2020-08-22,2019-2020
2,294787,270557,Leinster,55,Glasgow Warriors,19,9.0,3.0,5.0,2.0,...,4.0,1.0,8.0,8.0,1.00,5.0,6.0,0.83,2020-02-28,2019-2020
3,294781,270557,Ospreys,13,Leinster,21,1.0,3.0,1.0,3.0,...,5.0,1.0,12.0,12.0,1.00,9.0,13.0,0.69,2020-02-21,2019-2020
4,294775,270557,Leinster,36,Cheetahs,12,5.0,2.0,4.0,1.0,...,4.0,1.0,11.0,12.0,0.91,10.0,11.0,0.90,2020-02-15,2019-2020


In [5]:
df_rev.columns

Index(['game_id', 'league_id', 'home_team', 'home_team_score', 'away_team',
       'away_team_score', 'home_tries', 'away_tries', 'home_conversions',
       'away_conversions', 'home_penalty_goals', 'away_penalty_goals',
       'home_kick_percent', 'away_kick_percent', 'home_total_meters',
       'away_total_meters', 'home_kfh', 'away_kfh', 'home_pass_meters',
       'away_pass_meters', 'home_runs', 'away_runs', 'home_possession_1h_2h',
       'home_territory_1h_2h', 'home_clean_breaks', 'home_defenders_beaten',
       'home_offloads', 'home_rucks_won', 'home_mauls_won',
       'home_turnovers_conceeded', 'away_possession_1h_2h',
       'away_territory_1h_2h', 'away_clean_breaks', 'away_defenders_beaten',
       'away_offloads', 'away_rucks_won', 'away_mauls_won',
       'away_turnovers_conceeded', 'home_total_possession',
       'home_total_territory', 'away_total_possesion', 'away_total_territory',
       'home_scrum', 'home_lineout', 'away_scrum', 'away_lineout',
       'home_tackle

In [8]:
grouped_df = urc_test.groupby('league_season')
df_dict = {name: group for name, group in grouped_df}


In [10]:
for key, value in df_dict.items():
    print(key)

2019-2020
2020-2021
2021-2022
2022-2023
2023-2024


In [9]:
df_dict['2019-2020']

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent,date_formatted,league_season
413,592889,270557,Ulster,10,Leinster,28,1.0,3.0,1.0,2.0,...,7.0,1.00,6.0,6.0,1.00,14.0,14.0,1.00,2020-08-29,2019-2020
414,592884,270557,Leinster,27,Munster,25,3.0,3.0,3.0,2.0,...,10.0,0.90,4.0,4.0,1.00,13.0,13.0,1.00,2020-08-22,2019-2020
415,294787,270557,Leinster,55,Glasgow Warriors,19,9.0,3.0,5.0,2.0,...,4.0,1.00,8.0,8.0,1.00,5.0,6.0,0.83,2020-02-28,2019-2020
416,294781,270557,Ospreys,13,Leinster,21,1.0,3.0,1.0,3.0,...,5.0,1.00,12.0,12.0,1.00,9.0,13.0,0.69,2020-02-21,2019-2020
417,294775,270557,Leinster,36,Cheetahs,12,5.0,2.0,4.0,1.0,...,4.0,1.00,11.0,12.0,0.91,10.0,11.0,0.90,2020-02-15,2019-2020
418,294771,270557,Leinster,54,Connacht,7,8.0,1.0,7.0,1.0,...,3.0,1.00,5.0,5.0,1.00,10.0,13.0,0.76,2020-01-04,2019-2020
429,592886,270557,Connacht,26,Ulster,20,4.0,2.0,3.0,2.0,...,3.0,0.66,13.0,13.0,1.00,11.0,15.0,0.73,2020-08-23,2019-2020
430,294784,270557,Ulster,20,Cheetahs,10,2.0,1.0,2.0,1.0,...,6.0,1.00,1.0,1.0,1.00,17.0,20.0,0.85,2020-02-22,2019-2020
431,294777,270557,Ospreys,26,Ulster,24,3.0,3.0,1.0,3.0,...,6.0,0.83,4.0,4.0,1.00,12.0,14.0,0.85,2020-02-15,2019-2020
432,294766,270557,Ulster,38,Munster,17,5.0,2.0,5.0,2.0,...,2.0,1.00,6.0,6.0,1.00,9.0,10.0,0.90,2020-01-03,2019-2020


In [6]:
df_dict.keys()

dict_keys(['2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024'])

In [4]:
urc_test.head()

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent,date_formatted,league_season
0,598075,270557,Munster,10,Glasgow Warriors,17,1.0,2.0,1.0,2.0,...,5.0,1.0,11.0,14.0,0.78,9.0,12.0,0.75,2024-06-15,2023-2024
1,598070,270557,Munster,23,Ospreys,7,2.0,1.0,2.0,1.0,...,2.0,1.0,5.0,7.0,0.71,9.0,10.0,0.90,2024-06-07,2023-2024
2,598069,270557,Munster,29,Ulster,24,4.0,3.0,3.0,3.0,...,3.0,1.0,5.0,5.0,1.00,7.0,9.0,0.77,2024-06-01,2023-2024
3,598055,270557,Edinburgh,26,Munster,29,2.0,4.0,2.0,3.0,...,2.0,1.0,4.0,4.0,1.00,8.0,10.0,0.80,2024-05-17,2023-2024
4,598052,270557,Munster,47,Connacht,12,7.0,2.0,6.0,1.0,...,1.0,1.0,6.0,8.0,0.75,7.0,10.0,0.70,2024-05-11,2023-2024


In [ ]:
run_joiner_output(urc_sched_files, urc_game_files, 'game_data_v2')
run_joiner_output(prem_sched_files, prem_game_files, 'game_data_v2')
run_joiner_output(prem_sched_files, prem_game_files, 'game_data_v2')


In [3]:
urc_sched_df = pd.concat(list(map(pd.read_csv, urc_sched_files)))
urc_game_df = pd.concat(list(map(pd.read_csv, urc_game_files)))

prem_sched_df = pd.concat(list(map(pd.read_csv, prem_sched_files)))
prem_game_df = pd.concat(list(map(pd.read_csv, prem_game_files)))

prem_sched_df = pd.concat(list(map(pd.read_csv, prem_sched_files)))
prem_game_df = pd.concat(list(map(pd.read_csv, prem_game_files)))

sr_sched_df = pd.concat(list(map(pd.read_csv, sr_sched_files)))
sr_game_df = pd.concat(list(map(pd.read_csv, sr_game_files)))


In [6]:
urc_game_df.columns

Index(['game_id', 'league_id', 'home_team', 'home_team_score', 'away_team',
       'away_team_score', 'home_tries', 'away_tries', 'home_conversions',
       'away_conversions', 'home_penalty_goals', 'away_penalty_goals',
       'home_kick_percent', 'away_kick_percent', 'home_total_meters',
       'away_total_meters', 'home_kfh', 'away_kfh', 'home_pass_meters',
       'away_pass_meters', 'home_runs', 'away_runs', 'home_possession_1h_2h',
       'home_territory_1h_2h', 'home_clean_breaks', 'home_defenders_beaten',
       'home_offloads', 'home_rucks_won', 'home_mauls_won',
       'home_turnovers_conceeded', 'away_possession_1h_2h',
       'away_territory_1h_2h', 'away_clean_breaks', 'away_defenders_beaten',
       'away_offloads', 'away_rucks_won', 'away_mauls_won',
       'away_turnovers_conceeded', 'home_total_possession',
       'home_total_territory', 'away_total_possesion', 'away_total_territory',
       'home_scrum', 'home_lineout', 'away_scrum', 'away_lineout',
       'home_tackle

In [6]:
urc_sched_df[['month', 'day']] = urc_sched_df['date'].apply(parse_game_date_v2).apply(pd.Series)
urc_sched_df[['date', 'month', 'day']]

urc_sched_df['date_formatted'] = urc_sched_df.apply(lambda x: datetime.datetime(x['season'], x['month'], x['day']), axis=1)
urc_sched_df[['date', 'month', 'day', 'season', 'date_formatted']]

,date,month,day,season,date_formatted
0,"Fri, Dec 3",12,3,2021,2021-12-03
1,"Sat, Nov 27",11,27,2021,2021-11-27
2,"Fri, Oct 22",10,22,2021,2021-10-22
3,"Sat, Oct 16",10,16,2021,2021-10-16
4,"Sat, Oct 9",10,9,2021,2021-10-09
...,...,...,...,...,...
131,"Sat, Oct 28",10,28,2023,2023-10-28
132,"Sat, Mar 25",3,25,2023,2023-03-25
133,"Sun, Jan 29",1,29,2023,2023-01-29
134,"Sat, Apr 22",4,22,2023,2023-04-22


In [7]:
urc_sched_df['league_season'] = urc_sched_df.apply(assign_season, axis=1)
urc_sched_df[['date', 'month', 'day', 'season', 'date_formatted', 'league_season']]

,date,month,day,season,date_formatted,league_season
0,"Fri, Dec 3",12,3,2021,2021-12-03,2021-2022
1,"Sat, Nov 27",11,27,2021,2021-11-27,2021-2022
2,"Fri, Oct 22",10,22,2021,2021-10-22,2021-2022
3,"Sat, Oct 16",10,16,2021,2021-10-16,2021-2022
4,"Sat, Oct 9",10,9,2021,2021-10-09,2021-2022
...,...,...,...,...,...,...
131,"Sat, Oct 28",10,28,2023,2023-10-28,2023-2024
132,"Sat, Mar 25",3,25,2023,2023-03-25,2022-2023
133,"Sun, Jan 29",1,29,2023,2023-01-29,2022-2023
134,"Sat, Apr 22",4,22,2023,2023-04-22,2022-2023
